In [ ]:
# 2) Quais são os 10 produtos mais vendidos (por quantidade total de itens vendidos)? Apresente também a receita gerada por cada um deles.

import pandas as pd
from pathlib import Path
from IPython.display import display

DATA_DIR = Path('../data')

def brl(valor: float) -> str: 
    return f"R$ {valor:,.2f}".replace(',', 'v').replace('.', ',').replace('v', '.')

def load_clean(tabela: str) -> pd.DataFrame:
    path = DATA_DIR / f'{tabela}_limpo.csv'
    if not path.exists(): raise FileNotFoundError("Erro: Rode o script ETL (Q6) primeiro.")
    return pd.read_csv(path)

df_itens    = load_clean('itens_pedido')
df_produtos = load_clean('produtos')

df_itens['receita_item'] = df_itens['quantidade'] * df_itens['preco_praticado']

agg = df_itens.groupby('produto_id').agg(
    quantidade    = ('quantidade',    'sum'),
    receita_total = ('receita_item',  'sum'),
).reset_index()

top10 = (
    agg.merge(
        df_produtos[['id', 'nome']],
        left_on='produto_id', right_on='id',
        how='inner', validate='many_to_one',
    )
    .sort_values('quantidade', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top10.insert(0, 'Posição', [f'{i+1}º' for i in top10.index])
top10['Receita Total'] = top10['receita_total'].apply(brl)
top10 = top10.rename(columns={'nome': 'Nome', 'quantidade': 'Quantidade'})

print("─── Top 10 Produtos Mais Vendidos ───")
display(top10[['Posição', 'Nome', 'Quantidade', 'Receita Total']].style.hide(axis='index'))

─── Top 10 Produtos Mais Vendidos ───


Posição,Nome,Quantidade,Receita Total
1º,Acessórios 43,673,"R$ 836.332,93"
2º,Tênis Esportivo 78,669,"R$ 2.017.790,51"
3º,Tênis Esportivo 75,663,"R$ 1.184.919,07"
4º,Notebooks 172,653,"R$ 1.671.241,16"
5º,Roupas Íntimas 48,651,"R$ 1.906.096,07"
6º,Vitaminas 111,649,"R$ 1.600.955,33"
7º,Cosméticos 92,642,"R$ 1.213.697,40"
8º,Chás 118,637,"R$ 1.765.601,55"
9º,Suplementos 194,635,"R$ 538.652,03"
10º,Quadros 65,633,"R$ 1.573.600,50"
